In [1]:
%pip install --upgrade openai pydantic mermaid-py

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 27.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 44.7 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.8/5.8 MB 40.0 MB/s  0:00:00
  Attempting uninstall: ruff
    Found existing installation: ruff 0.11.8
    Uninstalling ruff-0.11.8:
      Successfully uninstalled ruff-0.11.8
  Attempting uninstall: filelock
    Found existing installation: filelock 3.13.1
    Uninstalling filelock-3.13.1:
      Successfully uninstalled filelock-3.13.1
  Attempting uninstall: openai╸━━━━━━━━━━━━━━━━━━  6/11 [virtualenv]
    Found existing installation: openai 2.21.0━━━━━━━━━━━━━━━━  6/11 [virtualenv]
    Uninstalling openai-2.21.0:91m╸━━━━━━━━━━━━━━━━━━  6/11 [virtualenv]
      Successfully uninstalled openai-2.21.0━━━━━━━━━━━━━━━━━━  6/11 [virtualenv]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11/11 [mermaid-py]1 [openai]
ERROR: pip's dependency resolver does not currently take into account all th

# Blog Post Generator Agent: Planner, Executor, and Critic Loop

In this notebook, you will build a multi-agent orchestration system that uses a **Planner, Executor, and Critic** loop to iteratively produce high-quality blog posts. You will learn how to coordinate multiple AI models with different strengths through structured planning, fast generation, and rigorous critique with convergence detection.

## Architecture Overview

This agent system uses three specialized roles, each backed by a model chosen for its strengths:

| Role | Model | Why This Model? |
|------|-------|------------------|
| **Planner** | `o4-mini` (reasoning) | Deep reasoning produces well-structured, comprehensive article outlines |
| **Executor** | `gpt-5-mini` (cheap) | Fast, cost-effective bulk content generation from a clear plan |
| **Critic** | `o4-mini` (reasoning) | Rigorous evaluation requires reasoning about quality criteria |

The **Orchestrator** manages the loop, enforcing `MAX_ITERATIONS` to prevent runaway costs and detecting **convergence** when the critic's score exceeds a quality threshold.

**How it works:**
1. The **Planner** receives a topic and generates a detailed article outline.
2. The **Executor** writes the full article based on that plan.
3. The **Critic** evaluates the article, scores it, and provides actionable feedback.
4. If the score is below the threshold and iterations remain, the Executor revises using the feedback.
5. The loop repeats until the quality threshold is met or `MAX_ITERATIONS` is reached.

In [ ]:
import mermaid as md

md.Mermaid("""
flowchart LR
    A[Topic] --> B[Planner]
    B --> C[Plan]
    C --> D[Executor]
    D --> E[Draft]
    E --> F[Critic]
    F --> G[Feedback]
    G -- If not converged --> D
""")

## Key Concepts

### MAX_ITERATIONS: Preventing Infinite Loops
Every iterative agent loop **must** have a hard limit on revision cycles. Without this, a perfectionist critic could keep requesting changes forever, wasting time and money. We set `MAX_ITERATIONS = 3` as a sensible default.

### Convergence: Knowing When to Stop
Convergence means the output has reached an acceptable quality level. We define this as the critic's score exceeding `QUALITY_THRESHOLD` (e.g., 8 out of 10). When the article is "good enough," we stop iterating early rather than burning through all iterations.

### Cost Optimization: Right Model for the Right Job
Not every step needs the most powerful model. Reasoning models (`o4-mini`) excel at structured thinking, which makes them perfect for planning and critique. Bulk content generation, on the other hand, is better served by cheaper, faster models (`gpt-5-mini`). This keeps costs low while maintaining quality where it matters most.

### Separation of Concerns
Each role has a **single responsibility**:
- The Planner only plans (it never writes the article).
- The Executor only writes (it does not evaluate its own work).
- The Critic only evaluates (it does not rewrite the article).

This separation makes the system easier to debug, test, and improve. You can swap models, adjust prompts, or change scoring criteria for each role independently.

## Setup

Let's install the required packages and configure the OpenAI client. You will also define the Pydantic model that structures the Critic's output.

In [ ]:
import os
import json
from getpass import getpass
from pydantic import BaseModel, Field
from openai import OpenAI

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = os.environ.get("OPENAI_API_KEY")

client = OpenAI()

# Model configuration - use reasoning models for planning/critique, cheap model for writing
PLANNER_MODEL = "o4-mini"       # Reasoning model for high-quality planning
WRITER_MODEL = "gpt-5-mini"    # Cost-effective model for content generation
CRITIC_MODEL = "o4-mini"        # Reasoning model for quality evaluation
MAX_ITERATIONS = 3              # Maximum revision cycles
QUALITY_THRESHOLD = 8           # Score out of 10 to stop iterating


class ArticleCritique(BaseModel):
    """Structured critique from the reviewer."""
    score: int = Field(description="Quality score from 1 to 10")
    strengths: list[str] = Field(description="List of article strengths")
    weaknesses: list[str] = Field(description="List of article weaknesses")
    specific_feedback: str = Field(description="Detailed, actionable feedback for improvement")
    ready_to_publish: bool = Field(description="Whether the article is publication-ready")

## The Planner

The Planner is the first step in your pipeline. It uses a **reasoning model** (`o4-mini`) to generate a comprehensive article outline from a given topic.

Why a reasoning model? Planning requires structured thinking: organizing sections logically, anticipating what the audience needs, and ensuring complete coverage of the topic. This is exactly where reasoning models shine.

Notice how the plan is returned as **raw text** (not JSON), because the Executor needs a natural-language brief to write from, not a rigid data structure.

In [ ]:
def plan_article(topic, audience="general technical audience"):
    """
    Uses a reasoning model to generate a detailed article plan.
    Returns the plan as raw text (not structured JSON).
    """
    response = client.responses.create(
        model=PLANNER_MODEL,
        input=f"""Create a detailed blog post plan for the following topic.

Topic: {topic}
Target Audience: {audience}

Your plan should include:
1. A compelling title
2. An introduction hook
3. 3-5 main sections with key points for each
4. Code examples or practical demonstrations to include
5. A conclusion with key takeaways

Return the plan as clear, structured text. This plan will be given to a writer to produce the full article."""
    )

    print("=" * 60)
    print("ARTICLE PLAN")
    print("=" * 60)
    print(response.output_text)
    return response.output_text

### Test the Planner

Let's run the Planner on a sample topic and see what kind of outline it produces. Pay attention to how the reasoning model organizes the content into a logical structure.

In [ ]:
topic = "Building Reliable AI Agents: Lessons from Production Systems"
plan = plan_article(topic)

## The Executor / Writer

The Executor takes the Planner's outline and produces the full article. It uses a **cheap, fast model** (`gpt-5-mini`) because:

- The hard reasoning work (structure, coverage, flow) was already done by the Planner.
- The Executor just needs to "fill in" the content following a clear brief.
- Bulk text generation does not require expensive reasoning capabilities.

On subsequent iterations, the Executor also receives **feedback from the Critic** and incorporates it into the revision. This is how the article improves over each cycle.

In [ ]:
def write_article(plan, feedback=None):
    """
    Uses a cost-effective model to write the full article from a plan.
    Optionally incorporates feedback from the critic.
    """
    prompt = f"""Write a complete, publication-ready blog post based on this plan:

{plan}"""

    if feedback:
        prompt += f"""

IMPORTANT: The following feedback was provided by a reviewer. Address ALL points:

{feedback}"""

    prompt += """

Requirements:
- Write in a clear, engaging style suitable for a technical blog
- Include code examples where the plan specifies them
- Use markdown formatting (headers, code blocks, bullet points)
- Aim for 800-1200 words
- Make it practical and actionable"""

    response = client.responses.create(
        model=WRITER_MODEL,
        input=prompt
    )

    print(f"\nArticle generated ({len(response.output_text.split())} words)")
    return response.output_text

## The Critic

The Critic is the quality gatekeeper. It uses a **reasoning model** (`o4-mini`) to evaluate the article against the original plan and provide structured feedback.

Why a reasoning model for critique? Evaluation requires comparing the article against multiple criteria simultaneously, identifying subtle gaps in coverage, and formulating specific, actionable improvement suggestions. This is a reasoning-heavy task.

The Critic returns structured JSON with:
- A **score** (1-10) for convergence detection
- **Strengths** to reinforce what is working
- **Weaknesses** to identify what needs improvement
- **Specific feedback** that the Executor can act on directly

In [ ]:
def critique_article(article, plan, iteration):
    """
    Uses a reasoning model to evaluate the article and provide structured feedback.
    Returns an ArticleCritique Pydantic model with score and feedback.
    """
    response = client.responses.create(
        model=CRITIC_MODEL,
        input=f"""You are a senior technical editor reviewing a blog post.

ORIGINAL PLAN:
{plan}

ARTICLE (Iteration {iteration}):
{article}

Evaluate the article on these criteria:
1. Adherence to the plan (does it cover all planned sections?)
2. Technical accuracy
3. Writing quality (clarity, engagement, flow)
4. Practical value (actionable takeaways, useful code examples)
5. Completeness (introduction, body, conclusion)

Be rigorous but fair. Only give a score of 8+ if the article is genuinely publication-ready.""",
        text={
            "format": {
                "type": "json_schema",
                "name": "article_critique",
                "strict": True,
                "schema": ArticleCritique.model_json_schema()
            }
        }
    )

    critique = ArticleCritique.model_validate_json(response.output_text)

    print(f"\n{'='*60}")
    print(f"CRITIQUE (Iteration {iteration})")
    print(f"{'='*60}")
    print(f"Score: {critique.score}/10")
    print(f"Ready to publish: {critique.ready_to_publish}")
    print(f"\nStrengths:")
    for s in critique.strengths:
        print(f"  + {s}")
    print(f"\nWeaknesses:")
    for w in critique.weaknesses:
        print(f"  - {w}")
    print(f"\nFeedback: {critique.specific_feedback}")

    return critique

## The Orchestration Loop

This is the heart of the multi-agent system. The orchestrator coordinates the three roles in a loop:

1. **Plan once** : The Planner generates the outline (this only happens once).
2. **Write and critique iteratively** : The Executor writes, the Critic evaluates, and feedback flows back.
3. **Stop on convergence or max iterations** : The loop exits when the quality threshold is met OR when you have exhausted your iteration budget.

The two stopping conditions are critical:
- **`QUALITY_THRESHOLD`** (convergence): Stops early when the article is good enough, saving cost.
- **`MAX_ITERATIONS`** (safety net): Prevents infinite loops and unbounded spending.

In practice, most articles converge within 2 to 3 iterations.

In [ ]:
def generate_blog_post(topic, audience="general technical audience"):
    """
    Full Planner -> Executor -> Critic orchestration loop.

    The loop continues until either:
    1. The critic scores the article >= QUALITY_THRESHOLD
    2. MAX_ITERATIONS is reached
    """
    print(f"Starting blog post generation for: '{topic}'")
    print(f"Max iterations: {MAX_ITERATIONS}, Quality threshold: {QUALITY_THRESHOLD}/10")
    print(f"Models: Planner={PLANNER_MODEL}, Writer={WRITER_MODEL}, Critic={CRITIC_MODEL}")
    print("=" * 60)

    # Step 1: Plan
    print("\n[PHASE 1: PLANNING]")
    plan = plan_article(topic, audience)

    # Step 2: Write -> Critique -> Revise loop
    article = None
    feedback = None
    final_critique = None

    for iteration in range(1, MAX_ITERATIONS + 1):
        print(f"\n[PHASE 2: WRITING - Iteration {iteration}/{MAX_ITERATIONS}]")

        # Write (or revise) the article
        article = write_article(plan, feedback=feedback)

        # Critique the article
        print(f"\n[PHASE 3: CRITIQUE - Iteration {iteration}/{MAX_ITERATIONS}]")
        critique = critique_article(article, plan, iteration)
        final_critique = critique

        # Check convergence
        if critique.score >= QUALITY_THRESHOLD:
            print(f"\nQuality threshold reached! Score: {critique.score}/{QUALITY_THRESHOLD}")
            break
        elif iteration < MAX_ITERATIONS:
            print(f"\n-> Score {critique.score}/{QUALITY_THRESHOLD} - Revising...")
            feedback = critique.specific_feedback
        else:
            print(f"\nMax iterations reached. Final score: {critique.score}/10")

    return {
        "topic": topic,
        "plan": plan,
        "article": article,
        "final_score": final_critique.score,
        "iterations": iteration,
        "final_critique": final_critique
    }

## Run the Full Pipeline

Now let's run the entire Planner, Executor, and Critic loop end to end. Watch the output to see how each phase works and how the article improves with each iteration. You will see the Planner create a detailed outline, the Executor write a first draft, and the Critic score it with specific feedback.

In [ ]:
result = generate_blog_post(
    topic="Building Reliable AI Agents: Lessons from Production Systems",
    audience="software engineers building AI-powered applications"
)

## Display the Final Article

Let's render the final article with proper markdown formatting so you can see the finished product. Notice the final score and how many iterations it took to reach convergence.

In [ ]:
from IPython.display import Markdown, display

print(f"\nFinal Score: {result['final_score']}/10")
print(f"Iterations: {result['iterations']}/{MAX_ITERATIONS}")
print(f"\n{'='*60}")
print("FINAL ARTICLE")
print(f"{'='*60}\n")
display(Markdown(result['article']))

## Understanding the Cost Tradeoffs

One of the most important design decisions in multi-agent systems is **which model to use for each role**. Here is how the cost structure works in this pipeline:

| Role | Model | Cost Level | Justification |
|------|-------|------------|---------------|
| Planner | `o4-mini` | Higher | Planning requires structured reasoning to produce good outlines |
| Writer | `gpt-5-mini` | Lower | Content generation from a clear plan does not need deep reasoning |
| Critic | `o4-mini` | Higher | Quality evaluation requires comparing against multiple criteria simultaneously |

### Practical Implications

- **The loop typically converges in 2 to 3 iterations.** The first draft is often close, and one round of feedback usually addresses the major gaps.
- **The Planner only runs once.** This is a fixed cost regardless of how many iterations the write-critique loop takes.
- **Each iteration costs one Writer call + one Critic call.** So the variable cost is proportional to the number of iterations.

### Tuning for Your Use Case

Try adjusting these settings and see how they affect output quality and cost:

- **Prioritize quality:** Increase `QUALITY_THRESHOLD` to 9 and `MAX_ITERATIONS` to 5.
- **Prioritize speed/cost:** Lower `QUALITY_THRESHOLD` to 7 and `MAX_ITERATIONS` to 2.
- **Quick drafts:** Set `MAX_ITERATIONS = 1` to skip the revision loop entirely (plan + write + single critique).

## Experimenting with Different Settings

Let's try a completely different topic to see how the pipeline adapts. This demonstrates that the system generalizes across different types of content. Try changing the topic or audience to something you are interested in!

In [ ]:
# Try with a different topic
result_fast = generate_blog_post(
    topic="5 Python Tips Every Developer Should Know",
    audience="beginner Python programmers"
)

In [ ]:
# Display the second article
print(f"\nFinal Score: {result_fast['final_score']}/10")
print(f"Iterations: {result_fast['iterations']}/{MAX_ITERATIONS}")
print(f"\n{'='*60}")
print("FINAL ARTICLE")
print(f"{'='*60}\n")
# Add at top of cell: from IPython.display import display
display(Markdown(result_fast['article']))

## Your Turn: Build a Code Review Agent

Same Planner, Executor, and Critic pattern, but applied to code review instead of blog post writing. You will adapt what you just built to review code for bugs, security issues, and improvements.

## Exercise 1: Build a Code Review Agent with Planner, Executor, and Critic

In the Blog Post Generator notebook you saw how three specialized roles (Planner, Executor, Critic) collaborate in a loop to iteratively improve an article. In this exercise you will adapt that same pattern to **automated code review**.

Your agent will:
1. **Plan** the review by analyzing the code snippet and deciding what to look for (bugs, security issues, style problems, performance).
2. **Execute** the review by writing a detailed, structured assessment.
3. **Critique** the review itself, scoring its thoroughness and accuracy, then feeding improvement suggestions back to the executor.

The loop continues until the review quality exceeds `QUALITY_THRESHOLD` or `MAX_ITERATIONS` is reached.

This is the exact same orchestration loop from the blog post generator, just applied to a different domain. The key insight: the pattern is domain-agnostic.

In [ ]:
import mermaid as md

md.Mermaid("""
flowchart LR
    A[Code Snippet] --> B[Planner]
    B --> C[Review Plan]
    C --> D[Executor]
    D --> E[Review]
    E --> F[Critic]
    F --> G[Feedback]
    G -- If not converged --> D
""")

### Pydantic Models

The `CodeReview` model structures the executor's output. The `ReviewCritique` model structures the critic's evaluation. Both use `model_json_schema()` for structured outputs via the Responses API.

In [ ]:
class CodeReview(BaseModel):
    """Structured output from the code review executor."""
    score: int = Field(description="Quality score from 1 to 10")
    bugs_found: list[str] = Field(description="List of bugs or issues found")
    improvements: list[str] = Field(description="Suggested improvements")
    security_concerns: list[str] = Field(description="Any security issues")
    ready_for_merge: bool = Field(description="Whether the code is ready to merge")


class ReviewCritique(BaseModel):
    """Structured critique of the review itself."""
    score: int = Field(description="Review quality score from 1 to 10")
    strengths: list[str] = Field(description="What the review got right")
    weaknesses: list[str] = Field(description="What the review missed or got wrong")
    specific_feedback: str = Field(description="Actionable feedback to improve the review")

### Step 1: `plan_review(code_snippet)`

Use the reasoning model (`PLANNER_MODEL`) to analyze the code snippet and produce a review plan. The plan should identify what categories of issues to look for (SQL injection, file handling, error handling, naming, etc.) and prioritize them.

Return the plan as plain text, just like the blog post planner returned a plain-text outline.

In [ ]:
def plan_review(code_snippet):
    """Use a reasoning model to create a detailed review plan for the given code."""
    # YOUR CODE HERE
    # Hint: Use client.responses.create() with PLANNER_MODEL.
    # Ask the model to analyze the code and produce a prioritized checklist
    # of things to review (bugs, security, style, performance, error handling).
    pass

### Step 2: `execute_review(code_snippet, plan, feedback=None)`

Use the cost-effective model (`REVIEWER_MODEL`) to write the actual code review based on the plan. If `feedback` is provided (from a previous critique iteration), the reviewer should incorporate it.

Return a `CodeReview` Pydantic object by using structured outputs.

In [ ]:
def execute_review(code_snippet, plan, feedback=None):
    """Write a structured code review based on the plan, optionally incorporating feedback."""
    # YOUR CODE HERE
    # Hint: Build a prompt that includes the code snippet, the plan,
    # and (if present) the feedback from the critic.
    # Use client.responses.create() with REVIEWER_MODEL and structured output
    # via text={"format": {"type": "json_schema", ...}} with CodeReview.model_json_schema().
    # Return a CodeReview instance using CodeReview.model_validate_json(response.output_text).
    pass

### Step 3: `critique_review(review, code_snippet, plan, iteration)`

Use the reasoning model (`CRITIC_MODEL`) to evaluate how thorough and accurate the review is. The critic should check whether the review caught all the issues in the code, whether its suggestions are actionable, and whether anything was missed.

Return a `ReviewCritique` Pydantic object.

In [ ]:
def critique_review(review, code_snippet, plan, iteration):
    """Evaluate the quality of the code review and provide structured feedback."""
    # YOUR CODE HERE
    # Hint: Use client.responses.create() with CRITIC_MODEL and structured output.
    # The prompt should include the original code, the review plan, the review itself,
    # and the current iteration number.
    # Return a ReviewCritique instance.
    pass

### Step 4: `review_code(code_snippet)` (The Orchestration Loop)

This is the main function that ties everything together, exactly like `generate_blog_post()` from the blog post generator. It should:

1. Call `plan_review()` once.
2. Enter a loop up to `MAX_ITERATIONS`:
   a. Call `execute_review()` (passing feedback if available).
   b. Call `critique_review()` to evaluate the review.
   c. If the critique score meets `QUALITY_THRESHOLD`, break early (convergence).
   d. Otherwise, pass the critique feedback to the next iteration.
3. Return a dictionary with the final review, score, and iteration count.

In [ ]:
def review_code(code_snippet):
    """Full orchestration loop: Plan -> Review -> Critique -> Revise."""
    # YOUR CODE HERE
    # Hint: Follow the exact same structure as generate_blog_post() from the
    # blog post generator notebook.
    #
    # 1. plan = plan_review(code_snippet)
    # 2. Loop from 1 to MAX_ITERATIONS:
    #      review = execute_review(code_snippet, plan, feedback)
    #      critique = critique_review(review, code_snippet, plan, iteration)
    #      if critique.score >= QUALITY_THRESHOLD: break
    #      else: feedback = critique.specific_feedback
    # 3. Return {"review": review, "final_score": critique.score, "iterations": iteration}
    pass

### Test Your Code Review Agent

The sample code below has several intentional problems: SQL injection, plaintext password storage, missing error handling, and no connection cleanup. A thorough review should catch all of them.

In [ ]:
sample_code = '''
def process_user_data(data):
    conn = sqlite3.connect("users.db")
    query = f"SELECT * FROM users WHERE name = '{data[\'name\']}'"
    result = conn.execute(query).fetchall()
    password = data['password']
    with open('passwords.txt', 'a') as f:
        f.write(password)
    return result
'''

result = review_code(sample_code)

if result:
    print(f"\nFinal Score: {result['final_score']}/10")
    print(f"Iterations: {result['iterations']}/{MAX_ITERATIONS}")
    print(f"\nBugs found: {result['review'].bugs_found}")
    print(f"Security concerns: {result['review'].security_concerns}")
    print(f"Improvements: {result['review'].improvements}")
    print(f"Ready for merge: {result['review'].ready_for_merge}")
else:
    print("review_code() returned None. Implement the functions above and try again!")

## Summary

In this notebook, you built a complete **Planner, Executor, and Critic** multi-agent loop for blog post generation. Here are the key takeaways:

- **The Planner, Executor, and Critic pattern separates concerns for better quality.** Each role has a single, well-defined responsibility: the Planner structures, the Executor writes, and the Critic evaluates. This separation makes the system easier to debug, test, and improve.

- **Use reasoning models where reasoning matters; use cheap models for generation.** Reasoning models (`o4-mini`) are ideal for planning and critique because those tasks require structured thinking. Bulk content generation works well with cheaper models (`gpt-5-mini`), keeping costs manageable.

- **MAX_ITERATIONS prevents infinite loops. Always set a hard limit.** Without a maximum iteration count, a perfectionist critic could keep requesting revisions indefinitely. This is a safety net that bounds both time and cost.

- **QUALITY_THRESHOLD defines convergence. Stop when quality is "good enough."** Rather than always running all iterations, you stop early when the article meets your quality bar. This saves cost on articles that are strong from the first draft.

- **This pattern generalizes beyond blog posts.** The same Planner, Executor, and Critic loop can be applied to code generation, report writing, email drafting, marketing copy, or any task where iterative refinement improves output quality.

- **Each iteration should show measurable improvement via the critic's feedback.** The critic's structured feedback (score, strengths, weaknesses, specific suggestions) gives the Executor clear direction for improvement, making each revision cycle productive.